In [1]:
import sys 
print(sys.executable)

/Users/stevey/cpsc330arm/bin/python


In [2]:
# python -m pip install -r requirements.txt

import os
import logging
import calendar
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from typing import Iterable, Optional
from pybaseball import statcast


import pyarrow.parquet as pq


import numpy as np


import duckdb

import requests 
import json 

### STEPS
1. get data from pybaseball / MLB API 
2. Parquet on Google Drive 
3. DuckDB reads Parquet 
4. staging / processed / feature tables 

In [3]:
# Google Drive for parquet storage
# read root from .env file
load_dotenv()

logging.basicConfig(
    level=logging.INFO, # shows INFO, WARNING, and ERROR messages
    format="[%(asctime)s] %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

YEARS = range(2015, 2026)
MONTHS = [3, 4, 5, 6, 7, 8, 9, 10]
SOURCE_NAME = "statcast"

DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])
RAW_ROOT = DRIVE_ROOT / "data" / "raw" / SOURCE_NAME

DEFAULT_REDO = False
DEFAULT_CONTINUE_ON_ERROR = True


In [15]:
# check file status 
check_df = pd.DataFrame(
    columns=["year", "month", "exists", "nrows", "size_mb"]
)

ls = [] 
for year in YEARS:
    for month in MONTHS:
        filepath = RAW_ROOT / f"year={year}" / f"month={month:02d}" / f"statcast_{year}_{month:02d}.parquet"
        if filepath.exists():
            nrows = pq.ParquetFile(filepath).metadata.num_rows
            size_mb = filepath.stat().st_size / (1024 * 1024)
        else:
            nrows = 0
            size_mb = 0

        ls.append({
            "year": year,
            "month": month,
            "exists": filepath.exists(),
            "nrows": nrows,
            "size_mb": size_mb
        })

check_df = pd.DataFrame(ls).reset_index(drop=True)

print(f"total amount of rows (pitches): {check_df['nrows'].sum():,}")

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(check_df)

total amount of rows (pitches): 7,434,200


,year,month,exists,nrows,size_mb
0,2015,3,False,0,0.000000
1,2015,4,True,94253,10.131628
2,2015,5,True,121910,13.007297
3,2015,6,True,116165,12.430172
4,2015,7,True,108309,11.592992
5,2015,8,True,123528,13.137695
6,2015,9,True,121713,12.983788
7,2015,10,True,16428,1.962010
8,2016,3,False,0,0.000000
9,2016,4,True,104296,11.301394


In [31]:
# get to know all the columns 
# also check if theres any inconsistencies between the very first (2015 Apr) and the very last (2025 Sep) files
df_2015_4 = pq.read_table(RAW_ROOT / "year=2015" / "month=04" / "statcast_2015_04.parquet").to_pandas()
df_2025_9 = pq.read_table(RAW_ROOT / "year=2025" / "month=09" / "statcast_2025_09.parquet").to_pandas()

# compare columns 
cols_2015_4 = set(df_2015_4.columns)
cols_2025_9 = set(df_2025_9.columns)
print("columns in 2015 Apr but not in 2025 Sep:", cols_2015_4 - cols_2025_9) # set(); meaning they are consistent

print()
print(f"number of columns: {len(df_2025_9.columns)}")
# all columns 
# print 6 entries in a line then line break
for idx, i in enumerate(df_2025_9.columns):
    print(i, end="   /   ")
    if idx != 0 and idx % 6 == 0:
        print()

with pd.option_context("display.max_columns", None):
    display(df_2015_4.head())
    display(df_2025_9.head())

columns in 2015 Apr but not in 2025 Sep: set()

number of columns: 120
pitch_type   /   game_date   /   release_speed   /   release_pos_x   /   release_pos_z   /   player_name   /   batter   /   
pitcher   /   events   /   description   /   spin_dir   /   spin_rate_deprecated   /   break_angle_deprecated   /   
break_length_deprecated   /   zone   /   des   /   game_type   /   stand   /   p_throws   /   
home_team   /   away_team   /   type   /   hit_location   /   bb_type   /   balls   /   
strikes   /   game_year   /   pfx_x   /   pfx_z   /   plate_x   /   plate_z   /   
on_3b   /   on_2b   /   on_1b   /   outs_when_up   /   inning   /   inning_topbot   /   
hc_x   /   hc_y   /   tfs_deprecated   /   tfs_zulu_deprecated   /   umpire   /   sv_id   /   
vx0   /   vy0   /   vz0   /   ax   /   ay   /   az   /   
sz_top   /   sz_bot   /   hit_distance_sc   /   launch_speed   /   launch_angle   /   effective_speed   /   
release_spin_rate   /   release_extension   /   game_pk   /   fielder

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,year,month
0,SL,2015-04-30,84.7,-1.68,5.38,"Storen, Drew",608700,519322,strikeout,swinging_strike,<NA>,<NA>,<NA>,<NA>,14,Kevin Plawecki strikes out swinging.,R,R,R,NYM,WSH,S,2,None,0,2,2015,1.11,-0.78,1.051887,1.22717,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,150430_220556,3.780301,-124.114849,-0.35166,10.38,24.455,-39.941,3.53,1.57,<NA>,<NA>,<NA>,83.7,2570,5.9,413981,467092,475582,457787,488862,435622,150029,572191,547180,54.5,<NA>,0.0,0.0,1,0,0,<NA>,77,3,Slider,2,8,2,8,8,2,2,8,Standard,Standard,<NA>,-0.001,-0.157,<NA>,<NA>,<NA>,0.157,<NA>,-6,-6,0.001,0.001,27,24,28,24,1,3,2,1,2,1,4.0,-1.11,-1.11,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2015,4
1,SL,2015-04-30,83.1,-1.81,5.47,"Storen, Drew",608700,519322,None,swinging_strike,<NA>,<NA>,<NA>,<NA>,5,Swinging Strike,R,R,R,NYM,WSH,S,<NA>,None,0,1,2015,1.55,-0.32,0.261125,2.25018,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,150430_220536,1.193559,-121.905558,1.021625,14.641,24.091,-35.456,3.53,1.57,<NA>,<NA>,<NA>,81.8,2157,5.7,413981,467092,475582,457787,488862,435622,150029,572191,547180,54.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,77,2,Slider,2,8,2,8,8,2,2,8,Standard,Standard,<NA>,0.0,-0.056,<NA>,<NA>,<NA>,0.056,<NA>,-6,-6,0.001,0.001,27,24,28,24,1,3,2,1,2,1,3.67,-1.55,-1.55,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2015,4
2,FF,2015-04-30,94.5,-1.78,5.43,"Storen, Drew",608700,519322,None,called_strike,<NA>,<NA>,<NA>,<NA>,4,Called Strike,R,R,R,NYM,WSH,S,<NA>,None,0,0,2015,-0.98,1.15,-0.328601,2.899707,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,150430_220519,6.339401,-138.475421,-2.78718,-13.591,33.27,-17.374,3.75,1.57,<NA>,<NA>,<NA>,93.1,2286,6.0,413981,467092,475582,457787,488862,435622,150029,572191,547180,54.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,77,1,4-Seam Fastball,2,8,2,8,8,2,2,8,Standard,Standard,<NA>,0.0,-0.043,<NA>,<NA>,<NA>,0.043,<NA>,-6,-6,0.001,0.001,27,24,28,24,1,3,2,1,2,1,1.46,0.98,0.98,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2015,4
3,SI,2015-04-30,94.4,-1.71,5.32,"Storen, Drew",527038,519322,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,4,"Wilmer Flores grounds out, shortstop Ian Desmo...",R,R,R,NYM,WSH,X,6,ground_ball,0,1,2015,-1.33,0.12,-0.763004,2.513263,<NA>,<NA>,<NA>,1,9,Bot,108.75,155.51,<NA>,<NA>,<NA>,150430_220436,5.939225,-138.416074,-0.831684,-17.811,31.78,-30.531,3.58,1.56,9,86.1,-15,94.1,1907,6.5,413981,467092,475582,457787,488862,435622,150029,572191,547180,54.5,0.109,0.1,0.0,1,0,0,2,76,2,Sinker,2,8,2,8,8,2,2,8,Standard,Standard,<NA>,0.0,-0

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,year,month
0,FF,2025-09-28,95.7,-2.15,5.21,"Weissert, Greg",678009,669711,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,1,Parker Meadows flies out to left fielder Jarre...,R,L,R,BOS,DET,X,7,fly_ball,0,0,2025,-0.71,0.93,-0.395886,3.083176,<NA>,700242,628451,2,9,Top,96.06,86.56,<NA>,<NA>,<NA>,<NA>,6.169368,-139.262184,-2.088523,-10.601017,31.025947,-19.832498,3.45,1.69,290,83.8,37,94.9,2370,5.8,776151,657136,663993,666152,681987,686765,680776,678882,677800,54.67,0.01,0.009,0.0,1,0,0,3,74,1,4-Seam Fastball,4,3,3,4,3,4,3,4,Infield shade,Strategic,226,0.146,-0.282,69.0,7.1,0.016,0.282,88.0,1,-1,0.854,0.146,30,25,30,26,1,4,2,1,4,2,1.56,0.71,-0.71,20.9,5.991833,-1.319512,28.782516,41.559201,30.599805,2025,9
1,FF,2025-09-28,95.1,-1.91,5.1,"Weissert, Greg",668670,669711,strikeout,called_strike,<NA>,<NA>,<NA>,<NA>,6,Jake Rogers called out on strikes.,R,R,R,BOS,DET,S,2,None,0,2,2025,-0.93,0.93,0.567564,2.59319,<NA>,700242,628451,1,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.617368,-138.217784,-3.023825,-13.845463,30.454633,-19.764496,3.372472,1.526115,<NA>,<NA>,<NA>,95.6,2309,6.6,776151,657136,663993,666152,681987,686765,680776,678882,677800,53.86,<NA>,0.0,0.0,1,0,0,<NA>,73,4,4-Seam Fastball,4,3,3,4,3,4,3,4,Standard,Standard,229,0.086,-0.16,<NA>,<NA>,<NA>,0.16,<NA>,1,-1,0.768,0.232,30,30,30,30,1,3,2,9,4,9,1.59,0.93,0.93,20.5,<NA>,<NA>,<NA>,<NA>,<NA>,2025,9
2,FF,2025-09-28,95.4,-1.99,5.22,"Weissert, Greg",668670,669711,None,foul,<NA>,<NA>,<NA>,<NA>,2,Foul,R,R,R,BOS,DET,S,<NA>,None,0,2,2025,-0.85,1.14,0.05528,3.301461,<NA>,700242,628451,1,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,7.213709,-138.740928,-1.962713,-12.54588,30.045431,-17.188392,3.47,1.6,246,80.3,45,94.6,2314,5.8,776151,657136,663993,666152,681987,686765,680776,678882,677800,54.75,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,3,4-Seam Fastball,4,3,3,4,3,4,3,4,Standard,Standard,211,0.0,0.0,63.5,6.2,<NA>,0.0,88.0,1,-1,0.768,0.232,30,30,30,30,1,3,2,9,4,9,1.36,0.85,0.85,22.9,2.871131,31.805044,22.266527,37.478847,15.582717,2025,9
3,SL,2025-09-28,84.8,-2.33,4.72,"Weissert, Greg",668670,669711,None,swinging_strike,<NA>,<NA>,<NA>,<NA>,5,Swinging Strike,R,R,R,BOS,DET,S,<NA>,None,0,1,2025,0.32,0.62,0.14213,2.594828,<NA>,700242,628451,1,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,5.156934,-123.459684,0.134447,2.225813,24.4588,-26.088975,3.47,1.6,<NA>,<NA>,<NA>,85.0,2528,6.4,776151,657136,663993,666152,681987,686765,680776,678882,677800,54.14,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>

### Statcast search CSV documentation
link: https://baseballsavant.mlb.com/csv-docs
- release_pose_x and -z (ft) are measured from the catcher's perspective (so negative for RHP, different from Rapsodo Pitching!)
- player_name is (likely to be) pitcher's name; otherwise player ID is used 
- game_type (in our data only 'R' is selected):  
    - E = Exhibition, S = Spring Training, R = Regular Season, F = Wild Card, D = Divisional Series, L = League Championship Series, W = World Series
- stand is batter handedness; p_throws is pitcher handedness 
- type only has three values: B = ball, S = strike, X = in play 
- hit location: position of first fielder to touch the ball 
- pfx_x and _z: (horizontal/vertical) movement in ft from the catcher's perspective 
- plate_x and _z: position of the ball when it crosses home plate from the catcher's perspective 
- hc_x and _y: hit coordinate of batted ball 
- fielder_2: catcher 
- umpire is deprecated 
- vx/y/z0: velo in ft/s, determined at y=50ft 
- ax/y/z: accelration in ft/s/s also at y=50ft
- sz_top and _bot: top/bottom of the batter's strike zone set by the operator when the ball is halfway to the plate 


In [ ]:
# DuckDB for local databasing 
con = duckdb.connect("local_db/bayesball.duckdb")


# TODO: insert path for read_parquet 
con.execute("""
    CREATE OR REPLACE VIEW raw_statcast AS 
    SELECT * 
    FROM read_parquet('.')
""")
